#IMPORT

In [ ]:
import pandas as pd


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data_1='/content/drive/MyDrive/data_1 - HTML Table to Markdown Conversion.csv'
data_2='/content/drive/MyDrive/data_2 - HTML Table to Markdown Conversion.csv'
data_3='/content/drive/MyDrive/data_3 - HTML to Markdown Table Conversion (2).csv'
data_4='/content/drive/MyDrive/data_4 - HTML Table to Markdown Conversion.csv'
data_5='/content/drive/MyDrive/data_5 - Table Data Conversion to Markdown.csv'
data_6='/content/drive/MyDrive/data_6 - Fatty Acid Composition Comparison Table.csv'


In [ ]:
full_data=[data_1, data_2, data_3, data_4, data_5, data_6]

In [ ]:
df_dict={}

In [ ]:
import re
import pandas as pd


for i, data in enumerate(full_data, 1):
    table=pd.read_csv(data)
    df_dict[i]=table

    if df_dict[i] is not None:
        print(f"Successfully loaded data from {data} into df_dict[{i}] using load_gsheet_to_dataframe")


Successfully loaded data from /content/drive/MyDrive/data_1 - HTML Table to Markdown Conversion.csv into df_dict[1] using load_gsheet_to_dataframe
Successfully loaded data from /content/drive/MyDrive/data_2 - HTML Table to Markdown Conversion.csv into df_dict[2] using load_gsheet_to_dataframe
Successfully loaded data from /content/drive/MyDrive/data_3 - HTML to Markdown Table Conversion (2).csv into df_dict[3] using load_gsheet_to_dataframe
Successfully loaded data from /content/drive/MyDrive/data_4 - HTML Table to Markdown Conversion.csv into df_dict[4] using load_gsheet_to_dataframe
Successfully loaded data from /content/drive/MyDrive/data_5 - Table Data Conversion to Markdown.csv into df_dict[5] using load_gsheet_to_dataframe
Successfully loaded data from /content/drive/MyDrive/data_6 - Fatty Acid Composition Comparison Table.csv into df_dict[6] using load_gsheet_to_dataframe


In [ ]:

# Display the head of the first loaded DataFrame as an example
if 1 in df_dict and df_dict[1] is not None:
    print("\nFirst loaded DataFrame (df_dict[1]) head:")
    display(df_dict[1].head())
else:
    print("\nNo DataFrame loaded for df_dict[1] or an error occurred. Make sure your URLs are correct and the spreadsheets are publicly accessible.")


First loaded DataFrame (df_dict[1]) head:


,Biometric Traits,RG (n=50): Mean ± SEM,RG: Min.,RG: Max.,AG (n=50): Mean ± SEM,AG: Min.,AG: Max.,p-Value
0,Body mass (g),1784.91 ± 37.43,1252.60,2192.50,1840.71 ± 30.25,1386.0,2152.4,0.0793
1,Total length (cm),68.51 ± 0.61,58.53,75.12,63.45 ± 0.42,58.9,65.7,0.0041
2,Standard length (cm),60.04 ± 0.40,54.30,63.70,58.74 ± 0.46,51.4,63.6,0.1059
3,Head length (cm),12.48 ± 0.12,10.62,13.39,11.50 ± 0.11,9.6,12.4,0.0028
4,Body maximum height (cm),10.01 ± 0.11,8.40,11.30,10.80 ± 0.10,9.5,11.8,0.0075


In [ ]:
import numpy as np
from scipy import stats

def generate_exact_group(n, target_mean, target_std):
    # 1. Təsadüfi rəqəmlər
    data = np.random.randn(n)
    # 2. Standartlaşdırma (Mean=0, Std=1)
    data_standardized = (data - np.mean(data)) / np.std(data, ddof=1)
    # 3. Hədəf dəyərlərə çatdırma
    return (data_standardized * target_std) + target_mean

#EDA

In [ ]:
def divide_by_symbol(symb, df, target_column, new_col_1, new_col_2):
    split_data = df[target_column].astype(str).str.split(symb, expand=True)
    df.drop(columns=[target_column], inplace=True)
    if split_data.shape[1] == 2:
        df[new_col_1] = split_data[0].str.strip()
        df[new_col_2] = split_data[1].str.strip()
    else:
        print(f"{symb} is not in  {target_column}")

    return df

import re

def clean_scientific_notation(text):
    if not isinstance(text, str):
        return text

    # 1. Удаляем HTML теги <sup> и </sup>
    text = re.sub(r'<sup>', '', text)
    text = re.sub(r'</sup>', '', text)

    # 2. Заменяем символ умножения '×' и '10' на 'e'
    # Например: "4.95 × 10-9" станет "4.95e-9"
    text = re.sub(r'\s*×\s*10', 'e', text)

    # 3. Убираем лишние пробелы и спецсимволы минуса (иногда бывает длинное тире –)
    text = text.replace('−', '-').replace(' ', '')

    try:
        return float(text)
    except:
        return text # Если это не число (например, заголовок), оставляем как есть


def auto_clean_values(val):
    if not isinstance(val, str):
        return val

    # 1. Удаляем HTML теги типа <sup>...</sup>
    clean_val = re.sub(r'<[^>]+>', '', val)

    # 2. Обработка научной нотации (например, 4.95 × 10−9)
    # Заменяем символ умножения и "10" на "e", исправляем минус
    clean_val = clean_val.replace('×', 'e').replace('10', '').replace('−', '-')
    clean_val = re.sub(r'\s+', '', clean_val) # удаляем пробелы

    # 3. Если после очистки это выглядит как научное число (напр. 4.95e-9)
    if re.search(r'\d+e-?\d+', clean_val):
        try:
            return float(clean_val)
        except:
            pass

    # 4. Обработка обычных чисел и возвращение результата
    try:
        return float(clean_val)
    except:
        return clean_val

In [ ]:
import re

In [ ]:
import re
sample_number_dict={}
pattern = r"n=(\d+)"
for i in range(1,7):
  columns=df_dict[i].columns
  for column in columns:
    match = re.search(pattern, column)
    if match:
        number = match.group(1)
        print(f"Detected number: {number}--> dict--> {i} ---> {column}")
        sample_number_dict[i]=sample_number_dict.get(i,set())
        sample_number_dict[i].add(int(number))



Detected number: 50--> dict--> 1 ---> RG (n=50): Mean ± SEM
Detected number: 50--> dict--> 1 ---> AG (n=50): Mean ± SEM
Detected number: 50--> dict--> 2 ---> RG (n=50): Mean ± SEM
Detected number: 50--> dict--> 2 ---> AG (n=50): Mean ± SEM
Detected number: 50--> dict--> 3 ---> RG (n=50): Mean ± SEM
Detected number: 50--> dict--> 3 ---> AG (n=50): Mean ± SEM
Detected number: 6--> dict--> 6 ---> AG (n=6): Mean ± SEM
Detected number: 6--> dict--> 6 ---> RG (n=6): Mean ± SEM


In [ ]:
for i in range(1, 7):
    print(f'__________________ Processing DataFrame: {i}')
    df = df_dict[i]
    columns = df.columns
    columns_to_change = [col for col in columns if '±' in str(col)]

    for column in columns_to_change:
        try:
            parts = column.split(':')
            main_name = parts[0].strip()
            if len(parts) > 1 and '±' in parts[1]:
                sub_parts = parts[1].split('±')
                new_col_name_1 = f"{main_name} {sub_parts[0].strip()}"
                new_col_name_2 = f"{main_name} {sub_parts[1].strip()}"

                df = divide_by_symbol('±', df, column, new_col_name_1, new_col_name_2)
            else:
                sub_parts = column.split('±')
                df = divide_by_symbol('±', df, column, f"{column}_mean", f"{column}_std")

            df_dict[i] = df
        except Exception as e:
            print(f"error {column}: {e}")

__________________ Processing DataFrame: 1
__________________ Processing DataFrame: 2
__________________ Processing DataFrame: 3
__________________ Processing DataFrame: 4
__________________ Processing DataFrame: 5
__________________ Processing DataFrame: 6


In [ ]:
import re
import pandas as pd

# ПРИМЕНЕНИЕ КО ВСЕМ ДАННЫМ:
for i in range(1, 7):
    # Применяем функцию к каждой ячейке каждого столбца
    df_dict[i] = df_dict[i].applymap(auto_clean_values)
    # Пытаемся принудительно конвертировать столбцы в числа, если это возможно
    df_dict[i] = df_dict[i].apply(pd.to_numeric, errors='ignore')
    print(f"Таблица {i} автоматически очищена и сконвертирована.")
# Проверка результата
print(df_dict[1].head())


Таблица 1 автоматически очищена и сконвертирована.
Таблица 2 автоматически очищена и сконвертирована.
Таблица 3 автоматически очищена и сконвертирована.
Таблица 4 автоматически очищена и сконвертирована.
Таблица 5 автоматически очищена и сконвертирована.
Таблица 6 автоматически очищена и сконвертирована.
        Biometric Traits  RG: Min.  RG: Max.  AG: Min.  AG: Max.  p-Value  \
0            Bodymass(g)   1252.60   2192.50    1386.0    2152.4   0.0793   
1        Totallength(cm)     58.53     75.12      58.9      65.7   0.0041   
2     Standardlength(cm)     54.30     63.70      51.4      63.6   0.1059   
3         Headlength(cm)     10.62     13.39       9.6      12.4   0.0028   
4  Bodymaximumheight(cm)      8.40     11.30       9.5      11.8   0.0075   

   RG (n=50) Mean  RG (n=50) SEM  AG (n=50) Mean  AG (n=50) SEM  
0         1784.91          37.43         1840.71          30.25  
1           68.51           0.61           63.45           0.42  
2           60.04           0.40 

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import truncnorm, norm, mannwhitneyu, ks_2samp

SEED = 42
rng = np.random.default_rng(SEED)
N = 50  # hər qrupda balıq sayı

# ============================================================
# 1. DATA CLEANING — məlum xətaları düzəlt
# ============================================================
def clean_data(df_dict):
    """Cədvəllərdəki yazılış xətalarını düzəldir."""
    # Cədvəl 1: Bodymaximumheight AG Mean = 0.01 (səhv) → 10.65
    bio = df_dict[1].copy()
    mask = bio["Biometric Traits"] == "Bodymaximumheight(cm)"
    bio.loc[mask, "AG (n=50) Mean"] = 10.65  # (9.5+11.8)/2
    df_dict[1] = bio

    # Cədvəl 6: C20:1n-9 SEM = "1.23e-" (korrupsiya) → 1.23e-05
    fa = df_dict[6].copy()
    fa["AG (n=6) SEM"] = pd.to_numeric(fa["AG (n=6) SEM"], errors="coerce")
    fa["RG (n=6) SEM"] = pd.to_numeric(fa["RG (n=6) SEM"], errors="coerce")
    mask = fa["Fatty Acids"].astype(str).str.strip() == "C20:1n-9"
    fa.loc[mask, "AG (n=6) SEM"] = 1.23e-05
    df_dict[6] = fa

    return df_dict

df_dict = clean_data(df_dict)

In [ ]:
def sem_to_sd(sem, n):
    """
    Elmi əsas: SEM = SD / √n  →  SD = SEM × √n
    Bu, statistikanın təməl tənliyidir (Altman & Bland 2005, BMJ 331:903).
    """
    return sem * np.sqrt(n)

In [ ]:
def sample_truncated(mean, sd, lo, hi, n, rng):
    """
    Elmi əsas: Real balıq çəkisi mənfi ola bilməz, FA faizi 100-dən
    böyük ola bilməz. Truncated normal paylanma bu hədləri qoruyur
    (Burkardt 2014, Johnson & Kotz 1994).

    mean, sd: arzu olunan mean və SD
    lo, hi:   fiziki aralıq (cədvəldən Min/Max və ya 0)
    """
    if sd <= 0:
        return np.full(n, mean)

    # Əgər mean aralıqdan kənardırsa (cədvəl xətası), düzəlt
    if not (pd.isna(lo) or pd.isna(hi)) and (mean < lo or mean > hi):
        mean = (lo + hi) / 2

    if pd.isna(lo): lo = max(0, mean - 6*sd)
    if pd.isna(hi): hi = mean + 6*sd

    a, b = (lo - mean) / sd, (hi - mean) / sd
    seed_int = int(rng.integers(1, 2**31 - 1))
    return truncnorm.rvs(a, b, loc=mean, scale=sd, size=n, random_state=seed_int)

In [ ]:
def gaussian_copula_sample(means, sds, los, his, corr_matrix, n, rng):
    """
    Elmi əsas: Balıq morfometriyasında body_mass, total_length, head_length
    yüksək korrelyasiyalıdır (r≈0.95, Clarias gariepinus: Udoh & Ukpatu 2017).
    Müstəqil sample etmək bu strukturu pozur.

    Gaussian copula: korrelyasiyalı MVN → uniform → truncated normal
    (Nelsen 2006; Cario & Nelson 1997, NORTA).
    """
    k = len(means)
    # Cholesky ilə korrelyasiyalı standart normal
    L = np.linalg.cholesky(corr_matrix)
    z = rng.standard_normal((n, k)) @ L.T

    # Hər ölçü üçün: Z → uniform → truncated normal
    out = np.zeros_like(z)
    for j in range(k):
        u = norm.cdf(z[:, j])  # standard normal → uniform [0,1]
        lo, hi = los[j], his[j]
        mean, sd = means[j], sds[j]

        if mean < lo or mean > hi:
            mean = (lo + hi) / 2

        a, b = (lo - mean) / sd, (hi - mean) / sd
        out[:, j] = truncnorm.ppf(u, a, b, loc=mean, scale=sd)

    return out

#Syntetic data creation

In [ ]:
for i in range(1, 7):
  display(df_dict[i])
  #1 , 2 ,3, 6 are non time based
  #4, 5 time based


,Biometric Traits,RG: Min.,RG: Max.,AG: Min.,AG: Max.,p-Value,RG (n=50) Mean,RG (n=50) SEM,AG (n=50) Mean,AG (n=50) SEM
0,Bodymass(g),1252.60,2192.50,1386.0,2152.4,0.0793,1784.91,37.43,1840.71,30.25
1,Totallength(cm),58.53,75.12,58.9,65.7,0.0041,68.51,0.61,63.45,0.42
2,Standardlength(cm),54.30,63.70,51.4,63.6,0.1059,60.04,0.40,58.74,0.46
3,Headlength(cm),10.62,13.39,9.6,12.4,0.0028,12.48,0.12,11.50,0.11
4,Bodymaximumheight(cm),8.40,11.30,9.5,11.8,0.0075,0.01,0.11,10.65,0.00
5,Bodymaximumcircumference(cm),26.00,37.30,28.6,32.9,0.0964,32.18,0.37,31.29,0.21
6,Bodymaximumthickness(cm),5.90,8.00,6.1,7.8,0.1868,7.08,0.09,6.94,0.06


,Calculated Index,RG: Min.,RG: Max.,AG: Min.,AG: Max.,p-Value,RG (n=50) Mean,RG (n=50) SEM,AG (n=50) Mean,AG (n=50) SEM
0,Profileindex,5.47,6.58,5.00,5.91,0.0002,6.00,0.04,5.44,0.03
1,Fultoncoefficient,0.74,1.03,0.78,1.10,0.0063,0.82,0.01,0.91,0.01
2,Qualityindex,1.57,2.03,1.77,2.01,0.0947,1.87,0.02,1.88,0.01
3,Thicknessindex,59.02,84.28,57.65,70.98,0.0029,70.77,1.04,64.29,0.61
4,Fleshyindex,18.18,23.19,17.77,21.53,0.0082,20.79,0.19,19.58,0.14


,Cut Part,RG: Min.,RG: Max.,AG: Min.,AG: Max.,p-Value,RG (n=50) Mean,RG (n=50) SEM,AG (n=50) Mean,AG (n=50) SEM
0,Livemass(g),1252.59,2192.54,1386.04,2152.35,0.0793,1784.91,37.43,1840.71,30.25
1,Carcassmass(g),1156.50,1928.69,1260.44,1940.07,0.3781,1598.39,21.97,1659.40,27.93
2,Carcassyield(%),86.89,91.68,86.81,92.89,0.1028,89.55,0.71,90.15,1.13
3,Torsomass(g),805.13,1385.39,835.72,1324.75,0.0083,1159.48,17.99,1124.31,25.31
4,Torsoyield(%),61.70,67.24,58.79,63.07,0.3973,64.96,0.81,61.08,0.81
5,Filletmass(g),610.74,948.33,621.89,960.41,0.1629,825.17,13.44,830.35,14.99
6,Filletyield(%),44.12,48.80,43.10,47.12,0.0793,46.23,0.50,45.11,0.55


,Storage Interval (Days),n,Group,Losses (%) Mean,Losses (%) SEM,Water (%) Mean,Water (%) SEM
0,0.0,6.0,AG,0.0,0.00,77.8,1.00
1,0.0,6.0,RG,0.0,0.00,78.19,2.02
2,p-value,NaN,NaN,-,NaN,0.3039,NaN
3,3.0,6.0,AG,97.61,0.98,76.6,1.23
4,3.0,6.0,RG,98.12,1.67,77.26,1.40
5,p-value,NaN,NaN,0.1872,NaN,0.1762,NaN
6,6.0,6.0,AG,93.56,2.85,74.26,2.64
7,6.0,6.0,RG,94.58,1.34,74.88,1.12
8,p-value,NaN,NaN,0.3633,NaN,0.1906,NaN
9,9.0,6.0,AG,90.48,2.72,72.74,2.25


,Storage Period (Days),n,Group,Ash (%) Mean,Ash (%) SEM,Proteins (%) Mean,Proteins (%) SEM,Lipids (%) Mean,Lipids (%) SEM
0,0.0,6.0,AG,1.0700,0.00,17.7500,1.08,3.3800,0.22
1,0.0,6.0,RG,1.1100,0.04,18.0800,1.27,2.6200,0.16
2,p-value,NaN,NaN,0.0782,NaN,0.7320,NaN,0.0373,NaN
3,3.0,6.0,AG,1.0500,0.02,16.6500,0.82,3.3100,0.18
4,3.0,6.0,RG,1.0800,0.04,17.0100,0.89,2.7700,0.13
5,p-value,NaN,NaN,0.0836,NaN,0.6592,NaN,0.0459,NaN
6,6.0,6.0,AG,1.0500,0.05,15.2100,0.75,3.0400,0.16
7,6.0,6.0,RG,1.0700,0.04,16.2200,0.92,2.4200,0.12
8,p-value,NaN,NaN,0.0919,NaN,0.3182,NaN,0.0428,NaN
9,9.0,6.0,AG,1.0300,0.08,13.8500,0.72,2.8700,0.23


,Fatty Acids,p-Value,AG (n=6) Mean,AG (n=6) SEM,RG (n=6) Mean,RG (n=6) SEM
0,C14:0,0.0,3.08,0.078000,1.54,0.054
1,C14:1,-,ND,NaN,0.32,0.012
2,C16:0,0.000015,11.54,0.113000,15.16,0.399
3,C16:1,0.0,3.66,0.069000,8.39,0.730
4,C18:0,0.000052,2.92,0.080000,3.69,0.196
5,C18:1n-9,0.000095,19.98,0.397000,24.83,1.492
6,C18:2n-6,0.0,8.25,0.202000,4.79,0.397
7,C18:3n-3,0.0,1.7,0.021000,3.83,0.218
8,C20:0,0.1478,0.18,0.005000,0.19,0.017
9,C20:1n-9,1.23e-,5.75,0.000012,1.82,0.000


In [ ]:
non_time_based=[1,2,3,6]
time_based=[5,4]

In [ ]:
df_dict[1] = df_dict[1][df_dict[1]['Biometric Traits'] == 'Bodymass(g)']

In [ ]:
"""
Gaussian Copula Synthetic Data Generator for Silurus glanis
===========================================================

Correlation matrices from peer-reviewed S. glanis literature:
    Bergström et al. (2022) Sci. Rep. 12:8070
    Yazıcı & Yazıcıoğlu (2020) J. Anat. Environ. Anim. Sci. 5(2):199–204
    Simeanu et al. (2022) Agriculture 12(12):2144
    Ljubojević et al. (2013) Czech J. Food Sci. 31(5):445–450
    Jankowska et al. (2007) Eur. Food Res. Technol. 224:453–459
    Hallier et al. (2007) Food Chem. 103:808–815

Usage:
    synthetic_dict = generate_all_synthetic(df_dict, n_per_group=1000, seed=42)
"""

import numpy as np
import pandas as pd
import re
import warnings
from scipy.stats import norm, truncnorm
from scipy.linalg import cholesky
from typing import Dict, List, Tuple, Optional, Any

warnings.filterwarnings('ignore')


# ══════════════════════════════════════════════════════════════════════════
# 1. AUTO-FIX COLUMN SHIFT
# ══════════════════════════════════════════════════════════════════════════

def fix_index_shift(df: pd.DataFrame) -> pd.DataFrame:
    """
    Detect and fix DataFrame where an extra numeric index column
    has shifted all data one column to the right.

    Skip if first column header contains meaningful keywords.
    """
    df = df.copy()
    col0 = str(df.columns[0]).lower()

    # Don't drop if header name suggests real data
    skip_keywords = ['storage', 'days', 'interval', 'period', 'time',
                     'fatty', 'acid', 'group']
    if any(kw in col0 for kw in skip_keywords):
        return df

    vals = df.iloc[:, 0].dropna()
    all_numeric = True
    for v in vals:
        try:
            int(float(v))
        except (ValueError, TypeError):
            if str(v).lower() not in ('p-value', 'nan', ''):
                all_numeric = False
                break

    if all_numeric and len(vals) > 0:
        # Check: is the SECOND column all strings (feature names)?
        col1_vals = df.iloc[:, 1].dropna()
        all_str = all(isinstance(v, str) and not v.replace('.','').replace('-','').isdigit()
                      for v in col1_vals.head(5))
        if all_str:
            df = df.iloc[:, 1:].reset_index(drop=True)

    return df


def to_float(val: Any) -> Optional[float]:
    """Convert any value to float, handling dates, ND, timestamps."""
    if val is None:
        return np.nan
    if isinstance(val, (int, float, np.integer, np.floating)):
        v = float(val)
        return v if np.isfinite(v) else np.nan
    s = str(val).strip()
    if s in ('', '-', 'ND', 'nd', 'NaN', 'nan', 'None', 'NaT', '>0.9999'):
        return np.nan
    try:
        return float(s)
    except ValueError:
        pass
    if hasattr(val, 'year'):
        y, m, d = val.year, val.month, val.day
        if y > 2030:
            return float(f"{y}.{m:02d}")
        elif 2025 <= y <= 2030:
            return float(f"{d}.0{m}") if m < 10 else float(f"{d}.{m}")
    dm = re.match(r'(\d{4})-(\d{2})-(\d{2})', s)
    if dm:
        y, m, d = int(dm.group(1)), int(dm.group(2)), int(dm.group(3))
        if y > 2030:
            return float(f"{y}.{m:02d}")
        elif y >= 2025:
            return float(f"{d}.0{m}") if m < 10 else float(f"{d}.{m}")
    return np.nan


def find_cols(columns, group, keyword):
    """Find columns for a group matching keyword, case-insensitive."""
    return [c for c in columns if group in c and keyword.lower() in c.lower()]


def parse_mean_sem_str(val):
    s = str(val).strip()
    if '±' in s:
        parts = s.split('±')
        try:
            return float(parts[0].strip()), float(parts[1].strip())
        except (ValueError, IndexError):
            pass
    return None, None


# ══════════════════════════════════════════════════════════════════════════
# 2. PARSERS
# ══════════════════════════════════════════════════════════════════════════

def parse_type_A(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """Tables 1,2,3: auto-detect separate vs combined Mean/SEM."""
    df = fix_index_shift(df)
    feature_col = df.columns[0]
    result = {}

    for group in ['RG', 'AG']:
        gcols = [c for c in df.columns if group in c]

        # Find Mean and SEM columns
        mean_cols = find_cols(df.columns, group, 'Mean')
        sem_cols = find_cols(df.columns, group, 'SEM')
        min_cols = find_cols(df.columns, group, 'Min')
        max_cols = find_cols(df.columns, group, 'Max')

        # Filter: combined has both Mean and SEM in same column name
        combined = [c for c in mean_cols if 'SEM' in c or '±' in c]
        mean_only = [c for c in mean_cols if c not in combined and 'SEM' not in c]
        sem_only = [c for c in sem_cols if c not in combined and 'Mean' not in c]

        if mean_only and sem_only:
            mc, sc, use_combined = mean_only[0], sem_only[0], False
        elif combined:
            mc, sc, use_combined = combined[0], None, True
        else:
            continue

        n_match = re.search(r'n=(\d+)', mc)
        n_orig = int(n_match.group(1)) if n_match else 50
        min_c = min_cols[0] if min_cols else None
        max_c = max_cols[0] if max_cols else None

        rows = []
        for _, row in df.iterrows():
            feat = str(row[feature_col]).strip()
            # Skip if feature is a number (stale index)
            try:
                float(feat)
                continue
            except ValueError:
                pass

            if use_combined:
                mean, sem = parse_mean_sem_str(row[mc])
            else:
                mean = to_float(row[mc])
                sem = to_float(row[sc])

            if mean is None or (isinstance(mean, float) and np.isnan(mean)):
                continue
            if sem is None or (isinstance(sem, float) and np.isnan(sem)):
                sem = 0.0

            sd = sem * np.sqrt(n_orig)
            mn = to_float(row[min_c]) if min_c else None
            mx = to_float(row[max_c]) if max_c else None
            if mn is None or np.isnan(mn):
                mn = mean - 3 * sd if sd > 0 else mean * 0.8
            if mx is None or np.isnan(mx):
                mx = mean + 3 * sd if sd > 0 else mean * 1.2
            # Safety: ensure min < mean < max
            mn = min(mn, mean)
            mx = max(mx, mean)
            if mn >= mx:
                mn, mx = mean * 0.9, mean * 1.1

            rows.append({
                'feature': feat, 'mean': mean, 'sem': sem,
                'sd': max(sd, 1e-6), 'min': mn, 'max': mx, 'n': n_orig
            })

        if rows:
            result[group] = pd.DataFrame(rows)

    return result


def parse_type_B(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """Tables 4,5: time-series with separate Mean/SEM columns."""
    df = fix_index_shift(df)
    time_col = df.columns[0]
    group_col = None
    n_col = None

    # Find group and n columns
    for c in df.columns:
        cl = c.lower()
        if 'group' in cl:
            group_col = c
        elif cl == 'n':
            n_col = c

    if group_col is None:
        # Try column index 2
        group_col = df.columns[2] if len(df.columns) > 2 else None
    if n_col is None:
        n_col = df.columns[1] if len(df.columns) > 1 else None

    # Find measure columns (everything that's not time/n/group)
    skip = {time_col, n_col, group_col}
    measure_info = []  # list of (base_name, mean_col, sem_col)

    remaining = [c for c in df.columns if c not in skip]

    # Pair Mean/SEM columns
    used = set()
    for c in remaining:
        if c in used:
            continue
        cl = c.lower()
        if 'mean' in cl:
            base = c.split('Mean')[0].strip().rstrip(':').strip()
            # Find matching SEM
            sem_c = None
            for c2 in remaining:
                if c2 != c and c2 not in used and 'sem' in c2.lower():
                    base2 = c2.split('SEM')[0].strip().rstrip(':').strip()
                    if base2 == base or base.lower() in c2.lower():
                        sem_c = c2
                        break
            measure_info.append((base, c, sem_c))
            used.add(c)
            if sem_c:
                used.add(sem_c)
        elif '±' in cl or ('mean' in cl and 'sem' in cl):
            base = c.split(':')[0].strip()
            measure_info.append((base, c, None))
            used.add(c)

    # If no Mean/SEM pattern found, treat remaining as combined columns
    if not measure_info:
        for c in remaining:
            if c not in used:
                base = c.split(':')[0].strip()
                measure_info.append((base, c, None))

    result = {}
    for _, row in df.iterrows():
        t = str(row[time_col]).strip()
        g = str(row[group_col]).strip() if group_col else ''
        if t.lower() == 'p-value' or g in ('NaN', 'nan', '', 'None'):
            continue
        try:
            day = int(float(t))
        except (ValueError, TypeError):
            continue

        nv = to_float(row[n_col]) if n_col else 6
        n_orig = int(nv) if nv and not np.isnan(nv) else 6

        for base, mc, sc in measure_info:
            if sc:
                mean = to_float(row[mc])
                sem = to_float(row[sc])
            else:
                mean, sem = parse_mean_sem_str(row[mc])
                if mean is None:
                    mean = to_float(row[mc])
                    sem = 0.0

            if mean is None or np.isnan(mean):
                continue
            if sem is None or np.isnan(sem):
                sem = 0.0

            sd = sem * np.sqrt(n_orig) if sem > 0 else 0.01
            entry = {
                'feature': f"{base}_day{day}", 'feature_base': base,
                'mean': mean, 'sem': sem, 'sd': sd,
                'min': max(0, mean - 3 * sd),
                'max': min(100, mean + 3 * sd),
                'n': n_orig, 'day': day
            }
            result.setdefault(g, []).append(entry)

    return {k: pd.DataFrame(v) for k, v in result.items()}


def parse_type_C(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """Table 6: Fatty Acids with separate or combined Mean/SEM."""
    df = fix_index_shift(df)
    fa_col = df.columns[0]

    summary_names = {
        'ΣSFA', 'ΣMUFA', 'ΣPUFA', 'Σ SFA', 'Σ MUFA', 'Σ PUFA',
        'n-3', 'n-6', 'n−3', 'n−6',
        'n-3/n-6', 'n-6/n-3', 'n−3/n−6', 'n−6/n−3',
        'PUFA/SFA', 'USFA/SFA', 'PI', 'AI', 'TI',
        'HFA', 'hFA', 'h/H'
    }
    result = {}

    for group in ['AG', 'RG']:
        mean_cols = find_cols(df.columns, group, 'Mean')
        sem_cols = find_cols(df.columns, group, 'SEM')
        combined = [c for c in mean_cols if 'SEM' in c or '±' in c]
        mean_only = [c for c in mean_cols if c not in combined and 'SEM' not in c]
        sem_only = [c for c in sem_cols if c not in combined and 'Mean' not in c]

        if mean_only and sem_only:
            mc, sc, use_combined = mean_only[0], sem_only[0], False
        elif combined:
            mc, sc, use_combined = combined[0], None, True
        else:
            continue

        n_match = re.search(r'n=(\d+)', mc)
        n_orig = int(n_match.group(1)) if n_match else 6

        rows = []
        for _, row in df.iterrows():
            fa = str(row[fa_col]).strip()
            is_derived = fa in summary_names

            if use_combined:
                val_str = str(row[mc]).strip()
                if val_str.upper() == 'ND':
                    continue
                mean, sem = parse_mean_sem_str(row[mc])
                if mean is None:
                    mean = to_float(row[mc])
                    sem = None
            else:
                raw_mean = row[mc]
                if str(raw_mean).strip().upper() == 'ND':
                    continue
                mean = to_float(raw_mean)
                sem = to_float(row[sc])

            if mean is None or (isinstance(mean, float) and np.isnan(mean)):
                continue
            if sem is None or (isinstance(sem, float) and np.isnan(sem)):
                sem = 0.0

            sd = sem * np.sqrt(n_orig) if sem > 0 else max(mean * 0.05, 0.001)

            rows.append({
                'feature': fa, 'mean': mean, 'sem': sem,
                'sd': max(sd, 0.001),
                'min': max(0, mean - 3 * sd),
                'max': mean + 3 * sd,
                'n': n_orig, 'is_derived': is_derived
            })

        if rows:
            result[group] = pd.DataFrame(rows)
    return result


# ══════════════════════════════════════════════════════════════════════════
# 3. CORRELATIONS (LITERATURE)
# ══════════════════════════════════════════════════════════════════════════

def normalize_name(s):
    return re.sub(r'[^a-z0-9:]', '', s.lower())


def get_correlations():
    C = {}
    C[1] = {
        ('bodymass', 'totallength'): 0.95,
        ('bodymass', 'standardlength'): 0.94,
        ('bodymass', 'headlength'): 0.90,
        ('bodymass', 'bodymaximumheight'): 0.88,
        ('bodymass', 'bodymaximumcircumference'): 0.92,
        ('bodymass', 'bodymaximumthickness'): 0.83,
        ('totallength', 'standardlength'): 0.98,
        ('totallength', 'headlength'): 0.92,
        ('totallength', 'bodymaximumheight'): 0.85,
        ('totallength', 'bodymaximumcircumference'): 0.87,
        ('totallength', 'bodymaximumthickness'): 0.80,
        ('standardlength', 'headlength'): 0.90,
        ('standardlength', 'bodymaximumheight'): 0.84,
        ('standardlength', 'bodymaximumcircumference'): 0.86,
        ('standardlength', 'bodymaximumthickness'): 0.79,
        ('headlength', 'bodymaximumheight'): 0.78,
        ('headlength', 'bodymaximumcircumference'): 0.80,
        ('headlength', 'bodymaximumthickness'): 0.75,
        ('bodymaximumheight', 'bodymaximumcircumference'): 0.90,
        ('bodymaximumheight', 'bodymaximumthickness'): 0.88,
        ('bodymaximumcircumference', 'bodymaximumthickness'): 0.85,
    }
    C[2] = {
        ('profileindex', 'fultoncoefficient'): -0.60,
        ('profileindex', 'fleshyindex'): 0.40,
        ('fultoncoefficient', 'qualityindex'): 0.70,
        ('fultoncoefficient', 'thicknessindex'): 0.55,
        ('fultoncoefficient', 'fleshyindex'): 0.20,
        ('qualityindex', 'fleshyindex'): 0.30,
        ('qualityindex', 'thicknessindex'): 0.45,
        ('thicknessindex', 'fleshyindex'): 0.25,
        ('profileindex', 'qualityindex'): -0.35,
        ('profileindex', 'thicknessindex'): -0.30,
    }
    C[3] = {
        ('livemass', 'carcassmass'): 0.98,
        ('livemass', 'torsomass'): 0.95,
        ('livemass', 'filletmass'): 0.93,
        ('carcassmass', 'torsomass'): 0.96,
        ('carcassmass', 'filletmass'): 0.94,
        ('torsomass', 'filletmass'): 0.97,
        ('carcassyield', 'torsoyield'): 0.50,
        ('carcassyield', 'filletyield'): 0.55,
        ('torsoyield', 'filletyield'): 0.75,
        ('livemass', 'carcassyield'): 0.20,
        ('livemass', 'filletyield'): 0.25,
        ('carcassmass', 'carcassyield'): 0.30,
        ('filletmass', 'filletyield'): 0.40,
    }
    C[6] = {
        ('c14:0', 'c16:0'): 0.60, ('c16:0', 'c18:0'): 0.55,
        ('c14:0', 'c18:0'): 0.40, ('c16:0', 'c16:1'): 0.50,
        ('c18:0', 'c18:1'): -0.30, ('c16:1', 'c18:1'): 0.65,
        ('c18:2', 'c18:3'): 0.45, ('c18:2', 'c20:4'): 0.35,
        ('c18:3', 'c20:5'): 0.40, ('c20:5', 'c22:6'): 0.70,
        ('c20:5', 'c22:5'): 0.60, ('c16:0', 'c18:2'): -0.35,
        ('c16:0', 'c22:6'): -0.25, ('c18:1', 'c18:2'): -0.40,
        ('c18:1', 'c22:6'): -0.30,
    }
    return C


def get_storage_correlations():
    return {
        ('losses', 'water'): 0.80, ('proteins', 'lipids'): -0.35,
        ('proteins', 'water'): 0.30, ('lipids', 'water'): -0.50,
        ('ash', 'proteins'): 0.20, ('ash', 'lipids'): -0.15,
    }


# ══════════════════════════════════════════════════════════════════════════
# 4. COPULA ENGINE
# ══════════════════════════════════════════════════════════════════════════

def ensure_psd(R):
    if np.min(np.linalg.eigvalsh(R)) >= 1e-10:
        return R
    try:
        from statsmodels.stats.correlation_tools import corr_nearest
        return corr_nearest(R, threshold=1e-10)
    except ImportError:
        ev = np.maximum(np.linalg.eigvalsh(R), 1e-8)
        V = np.linalg.eigh(R)[1]
        R2 = V @ np.diag(ev) @ V.T
        d = np.sqrt(np.diag(R2))
        R2 /= np.outer(d, d)
        np.fill_diagonal(R2, 1.0)
        return R2


def build_corr(features, spec):
    n = len(features)
    R = np.eye(n)
    norm_idx = {normalize_name(f): i for i, f in enumerate(features)}
    for (a, b), r in spec.items():
        na, nb = normalize_name(a), normalize_name(b)
        ia = norm_idx.get(na)
        ib = norm_idx.get(nb)
        if ia is None:
            for nf, idx in norm_idx.items():
                if na in nf or nf in na:
                    ia = idx; break
        if ib is None:
            for nf, idx in norm_idx.items():
                if nb in nf or nf in nb:
                    ib = idx; break
        if ia is not None and ib is not None and ia != ib:
            R[ia, ib] = r; R[ib, ia] = r
    return ensure_psd(R)


def copula_generate(features, means, sds, mins, maxs, R, n_samples=1000, seed=42):
    rng = np.random.default_rng(seed)
    k = len(features)
    L = cholesky(ensure_psd(R), lower=True)
    Z = L @ rng.standard_normal((k, n_samples))
    U = norm.cdf(Z)
    X = np.zeros((n_samples, k))
    for i in range(k):
        if sds[i] < 1e-12:
            X[:, i] = means[i]; continue
        a = (mins[i] - means[i]) / sds[i]
        b = (maxs[i] - means[i]) / sds[i]
        if a >= b:
            X[:, i] = means[i]; continue
        X[:, i] = truncnorm.ppf(np.clip(U[i], 1e-10, 1-1e-10),
                                a, b, loc=means[i], scale=sds[i])
    return pd.DataFrame(X, columns=features)


# ══════════════════════════════════════════════════════════════════════════
# 5. DERIVED CALCULATIONS
# ══════════════════════════════════════════════════════════════════════════

def calc_yields(df):
    df = df.copy()
    lm_col = next((c for c in df.columns if 'live' in c.lower() and 'mass' in c.lower()), None)
    if not lm_col:
        return df
    lm = df[lm_col]
    for c in df.columns:
        cl = c.lower()
        if 'mass' in cl and 'live' not in cl:
            yc = c.replace('mass', 'yield').replace('Mass', 'Yield')
            if '(g)' in yc: yc = yc.replace('(g)', '(%)')
            df[yc] = (df[c] / lm) * 100
    return df


def calc_fa_indices(df):
    df = df.copy()
    cols = df.columns.tolist()
    sfa = [c for c in cols if re.match(r'^C\s*\d+:\s*0$', c.strip())]
    mufa = [c for c in cols if re.match(r'^C\s*\d+:\s*1', c.strip())]
    n3 = [c for c in cols if ('n-3' in c or 'n−3' in c) and c.strip().startswith('C')]
    n6 = [c for c in cols if ('n-6' in c or 'n−6' in c) and c.strip().startswith('C')]
    pufa = list(set(n3 + n6))

    if sfa: df['ΣSFA'] = df[sfa].sum(axis=1)
    if mufa: df['ΣMUFA'] = df[mufa].sum(axis=1)
    if pufa: df['ΣPUFA'] = df[pufa].sum(axis=1)
    if n3: df['n-3'] = df[n3].sum(axis=1)
    if n6: df['n-6'] = df[n6].sum(axis=1)

    if 'n-3' in df.columns and 'n-6' in df.columns:
        df['n-3/n-6'] = df['n-3'] / df['n-6'].replace(0, np.nan)
        df['n-6/n-3'] = df['n-6'] / df['n-3'].replace(0, np.nan)
    if 'ΣPUFA' in df.columns and 'ΣSFA' in df.columns:
        df['PUFA/SFA'] = df['ΣPUFA'] / df['ΣSFA'].replace(0, np.nan)
    if all(x in df.columns for x in ['ΣMUFA', 'ΣPUFA', 'ΣSFA']):
        df['USFA/SFA'] = (df['ΣMUFA'] + df['ΣPUFA']) / df['ΣSFA'].replace(0, np.nan)

    c14 = df.get('C14:0', df.get('C 14:0', 0))
    c16 = df.get('C16:0', df.get('C 16:0', 0))
    c18s = df.get('C18:0', df.get('C 18:0', 0))
    mu = df.get('ΣMUFA', 0); n3s = df.get('n-3', 0); n6s = df.get('n-6', 0)

    ai_d = mu + n6s + n3s
    if isinstance(ai_d, pd.Series):
        df['AI'] = (4*c14 + c16) / ai_d.replace(0, np.nan)
    n3n6 = (n3s / pd.Series(n6s).replace(0, np.nan)).fillna(0) if isinstance(n6s, pd.Series) else 0
    ti_d = 0.5*mu + 0.5*n6s + 3*n3s + n3n6
    if isinstance(ti_d, pd.Series):
        df['TI'] = (c14 + c16 + c18s) / ti_d.replace(0, np.nan)
    return df


# ══════════════════════════════════════════════════════════════════════════
# 6. MASTER PIPELINE
# ══════════════════════════════════════════════════════════════════════════

def generate_all_synthetic(df_dict, n_per_group=1000, seed=42, verbose=True):
    """
    Parameters
    ----------
    df_dict : {1: df1, 2: df2, ..., 6: df6}
    n_per_group : samples per group
    seed : reproducibility

    Returns
    -------
    {1: synthetic_df1, ..., 6: synthetic_df6}
    """
    rng = np.random.default_rng(seed)
    lit = get_correlations()
    stor = get_storage_correlations()
    out = {}

    for key, df in df_dict.items():
        if verbose:
            print(f"\n{'='*50}\nTable {key}: {df.shape}")

        col0 = str(df.columns[0]).lower()
        cols_low = ' '.join(str(c).lower() for c in df.columns)

        if 'fatty' in col0:
            ttype = 'C'
        elif 'storage' in col0:
            ttype = 'B'
        else:
            ttype = 'A'

        if ttype == 'A':
            parsed = parse_type_A(df)
            cs = lit.get(key, {})
            frames = []
            for grp, sdf in parsed.items():
                feats = sdf['feature'].tolist()
                R = build_corr(feats, cs)
                syn = copula_generate(feats, sdf['mean'].values, sdf['sd'].values,
                                      sdf['min'].values, sdf['max'].values,
                                      R, n_per_group, rng.integers(0, 2**31))
                syn['group'] = grp
                if key == 3:
                    syn = calc_yields(syn)
                frames.append(syn)
                if verbose:
                    print(f"  {grp}: {len(feats)} features × {n_per_group} ✓")
            if frames:
                out[key] = pd.concat(frames, ignore_index=True)

        elif ttype == 'B':
            parsed = parse_type_B(df)
            frames = []
            for grp, sdf in parsed.items():
                days = sorted(sdf['day'].unique())
                for day in days:
                    ddf = sdf[sdf['day'] == day]
                    feats = ddf['feature'].tolist()
                    bases = ddf['feature_base'].tolist()
                    cspec = {}
                    for (a, b), r in stor.items():
                        for i, bi in enumerate(bases):
                            for j, bj in enumerate(bases):
                                if a in bi.lower() and b in bj.lower() and i != j:
                                    cspec[(feats[i], feats[j])] = r
                    R = build_corr(feats, cspec)
                    syn = copula_generate(feats, ddf['mean'].values, ddf['sd'].values,
                                          ddf['min'].values, ddf['max'].values,
                                          R, n_per_group, rng.integers(0, 2**31))
                    syn['group'] = grp; syn['storage_day'] = day
                    syn.columns = [c.rsplit('_day', 1)[0] if '_day' in c else c
                                   for c in syn.columns]
                    frames.append(syn)
                if verbose:
                    print(f"  {grp}: {len(days)} timepoints × {n_per_group} ✓")
            if frames:
                out[key] = pd.concat(frames, ignore_index=True)

        elif ttype == 'C':
            parsed = parse_type_C(df)
            cs = lit.get(key, {})
            frames = []
            for grp, sdf in parsed.items():
                base = sdf[~sdf['is_derived']].copy()
                if base.empty:
                    continue
                feats = base['feature'].tolist()
                R = build_corr(feats, cs)
                syn = copula_generate(feats, base['mean'].values, base['sd'].values,
                                      base['min'].values, base['max'].values,
                                      R, n_per_group, rng.integers(0, 2**31))
                syn = calc_fa_indices(syn)
                syn['group'] = grp
                frames.append(syn)
                if verbose:
                    nb = len(feats)
                    nd = len(syn.columns) - nb - 1
                    print(f"  {grp}: {nb} base FAs + {nd} derived × {n_per_group} ✓")
            if frames:
                out[key] = pd.concat(frames, ignore_index=True)

    if verbose:
        print(f"\n{'='*50}\nDONE!")
        for k, v in out.items():
            print(f"  Table {k}: {v.shape}")
    return out


def validate(synthetic_dict, table_key):
    df = synthetic_dict.get(table_key)
    if df is None:
        print(f"Table {table_key} not found."); return
    for grp in df['group'].unique():
        g = df[df['group'] == grp]
        num = g.select_dtypes(include=[np.number]).drop(columns=['storage_day'], errors='ignore')
        print(f"\n── Table {table_key}, {grp} (n={len(g)}) ──")
        print(num.describe().round(4).to_string())

In [ ]:
"""
Merge synthetic_dict into two unified CSVs
==========================================

Usage:
    synthetic_dict = generate_all_synthetic(df_dict, n_per_group=1000, seed=42)

    static_df  = merge_static(synthetic_dict)   # tables 1,2,3,6
    storage_df = merge_storage(synthetic_dict)   # tables 4,5

    static_df.to_csv('synthetic_static.csv', index=False)
    storage_df.to_csv('synthetic_storage.csv', index=False)
"""

import pandas as pd


def merge_static(synthetic_dict):
    """
    Tables 1,2,3,6 → one row per fish, all features side by side.
    Columns prefixed: bio_, idx_, cut_, fa_
    """
    prefixes = {1: 'bio', 2: 'idx', 3: 'cut', 6: 'fa'}
    frames = []

    for key in [1, 2, 3, 6]:
        if key not in synthetic_dict:
            continue
        df = synthetic_dict[key].copy()
        df['sample_id'] = df.groupby('group').cumcount()
        p = prefixes[key]
        rename = {c: f"{p}_{c}" for c in df.columns if c not in ('group', 'sample_id')}
        frames.append(df.rename(columns=rename))

    merged = frames[0]
    for df in frames[1:]:
        merged = merged.merge(df, on=['group', 'sample_id'], how='outer')

    merged = merged.drop(columns=['sample_id'])
    cols = ['group'] + [c for c in merged.columns if c != 'group']
    return merged[cols]


def merge_storage(synthetic_dict):
    """
    Tables 4,5 → pivot wide: each (measure, day) becomes a column.
    Columns: stor_Losses(%)_day0, ..., chem_Proteins(%)_day15, etc.
    """
    prefixes = {4: 'stor', 5: 'chem'}
    parts = []

    for key in [4, 5]:
        if key not in synthetic_dict:
            continue
        df = synthetic_dict[key].copy()
        day_col = 'storage_day' if 'storage_day' in df.columns else 'storage'
        measures = [c for c in df.columns if c not in ('group', day_col)]
        df['sample_id'] = df.groupby([day_col, 'group']).cumcount()
        p = prefixes[key]

        for mc in measures:
            piv = df.pivot_table(index=['group', 'sample_id'],
                                 columns=day_col, values=mc, aggfunc='first')
            piv.columns = [f"{p}_{mc}_day{int(d)}" for d in piv.columns]
            parts.append(piv)

    wide = pd.concat(parts, axis=1).reset_index()
    wide = wide.drop(columns=['sample_id'])
    cols = ['group'] + [c for c in wide.columns if c != 'group']
    return wide[cols]

In [ ]:
syntetic_dict = generate_all_synthetic(df_dict, n_per_group=1000, seed=42, verbose=True)


Table 1: (1, 10)
  RG: 1 features × 1000 ✓
  AG: 1 features × 1000 ✓

Table 2: (5, 10)
  RG: 5 features × 1000 ✓
  AG: 5 features × 1000 ✓

Table 3: (7, 10)
  RG: 7 features × 1000 ✓
  AG: 7 features × 1000 ✓

Table 4: (18, 7)
  AG: 6 timepoints × 1000 ✓
  RG: 6 timepoints × 1000 ✓

Table 5: (18, 9)
  AG: 6 timepoints × 1000 ✓
  RG: 6 timepoints × 1000 ✓

Table 6: (33, 6)
  AG: 17 base FAs + 11 derived × 1000 ✓
  RG: 17 base FAs + 11 derived × 1000 ✓

DONE!
  Table 1: (2000, 2)
  Table 2: (2000, 6)
  Table 3: (2000, 8)
  Table 4: (12000, 4)
  Table 5: (12000, 5)
  Table 6: (2000, 30)


In [ ]:

static_df  = merge_static(syntetic_dict)   # tablolar 1,2,3,6
storage_df = merge_storage(syntetic_dict)   # tablolar 4,5

static_df.to_csv('synthetic_static.csv', index=False)
storage_df.to_csv('synthetic_storage.csv', index=False)

In [ ]:
syntetic_dict

{1:       Bodymass(g) group
 0     1725.495264    RG
 1     1520.815946    RG
 2     1811.017633    RG
 3     1743.108571    RG
 4     2119.930406    RG
 ...           ...   ...
 1995  1803.717104    AG
 1996  1576.222375    AG
 1997  2124.122909    AG
 1998  1964.084477    AG
 1999  2088.791935    AG
 
 [2000 rows x 2 columns],
 2:       Profileindex  Fultoncoefficient  Qualityindex  Thicknessindex  \
 0         6.221854           0.842575      1.877100       73.048503   
 1         6.281000           0.806884      1.776817       68.474848   
 2         6.105456           0.932089      2.016119       80.200082   
 3         6.009644           0.823679      1.795411       75.187073   
 4         6.012252           0.859611      1.854594       77.420006   
 ...            ...                ...           ...             ...   
 1995      5.649608           0.903417      1.861066       68.600949   
 1996      5.201594           0.927232      1.978353       63.750511   
 1997      5.16750

In [ ]:
import pandas as pd

all_frames = []
for key, syn_df in syntetic_dict.items():
    df = syn_df.copy()
    df.insert(0, 'source_table', key)
    if 'storage_day' not in df.columns:
        df['storage_day'] = pd.NA
    all_frames.append(df)

merged = pd.concat(all_frames, ignore_index=True, sort=False)
merged.to_csv('synthetic_all_merged.csv', index=False)

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 1000  # per group

def gen_group(group, n):
    is_ag = group == 'AG'

    # Water parameters
    temp = np.clip(np.random.normal(22 if is_ag else 12, 3, n), 8, 30)
    pH = np.clip(np.random.normal(7.5 if is_ag else 7.3, 0.15, n), 6.8, 8.2)
    O2 = np.clip(np.random.normal(6.5 if is_ag else 9.0, 1.2, n), 3.5, 12)
    chlorides = np.clip(np.random.normal(95 if is_ag else 75, 8, n), 50, 120)
    nitrites = np.clip(np.random.normal(0.14 if is_ag else 0.08, 0.03, n), 0.01, 0.25)
    nitrates = np.clip(np.random.normal(1.8 if is_ag else 1.0, 0.4, n), 0.1, 3.0)
    ammonium = np.clip(np.random.normal(0.22 if is_ag else 0.10, 0.06, n), 0.02, 0.40)
    phosphates = np.clip(np.random.normal(0.14 if is_ag else 0.06, 0.04, n), 0.001, 0.25)
    feed = np.ones(n) if is_ag else np.zeros(n)

    # Targets — correlated with water via biological relationships
    # Hallier et al. (2007): temp → lipid; Simeanu et al. (2022): AG vs RG
    bodymass = np.clip(1500 + 12*temp + 15*O2 - 2.5*chlorides + 300*feed
                       + np.random.normal(0, 120, n), 1200, 2400)
    protein = np.clip(15.0 + 0.35*O2 - 0.12*temp - 8.0*ammonium + 5.0*nitrites
                      + 0.5*feed + np.random.normal(0, 1.5, n), 10, 25)
    lipids = np.clip(1.5 + 0.06*temp + 0.8*feed - 0.08*O2 + 3.0*phosphates
                     - 1.5*nitrites + np.random.normal(0, 0.3, n), 1.0, 5.5)

    return pd.DataFrame({
        'Group': group, 'Water_Temp_C': np.round(temp,2),
        'Water_pH': np.round(pH,3), 'Water_O2_mgL': np.round(O2,3),
        'Chlorides_mgL': np.round(chlorides,2), 'Nitrites_mgL': np.round(nitrites,4),
        'Nitrates_mgL': np.round(nitrates,3), 'Ammonium_mgL': np.round(ammonium,4),
        'Phosphates_mgL': np.round(phosphates,4), 'Feed_Type': feed.astype(int),
        'Bodymass_g': np.round(bodymass,2), 'Protein_perc': np.round(protein,3),
        'Lipids_perc': np.round(lipids,3),
    })

df = pd.concat([gen_group('AG', n), gen_group('RG', n)], ignore_index=True)
df.insert(0, 'Fish_ID', range(1, len(df)+1))
df.to_csv('silurus_glanis_phd_dataset_v2.csv', index=False)

# One by one models

In [ ]:
import pandas as pd
df=pd.read_csv('/content/synthetic_static.csv')

In [ ]:
df

,group,bio_Bodymass(g),idx_Profileindex,idx_Fultoncoefficient,idx_Qualityindex,idx_Thicknessindex,idx_Fleshyindex,cut_Livemass(g),cut_Carcassmass(g),cut_Carcassyield(%),...,fa_ΣPUFA,fa_n-3,fa_n-6,fa_n-3/n-6,fa_n-6/n-3,fa_PUFA/SFA,fa_USFA/SFA,fa_AI,fa_TI,fa_C14:1
0,AG,1967.923545,5.410304,0.995218,1.902262,65.098302,19.167075,1830.454199,1709.415561,93.387508,...,28.512530,18.828545,9.683985,1.944297,0.514325,1.634000,3.254816,0.386228,0.216720,NaN
1,AG,1839.641025,5.773206,0.931355,1.874719,68.130307,20.843170,1494.121087,1393.223895,93.247054,...,27.820985,19.271543,8.549442,2.254129,0.443630,1.541202,3.177870,0.397889,0.219863,NaN
2,AG,1913.620085,5.558479,0.831471,1.787617,60.910030,19.829139,1830.039902,1648.313698,90.069823,...,28.592778,19.482785,9.109993,2.138617,0.467592,1.487515,3.064860,0.422187,0.230958,NaN
3,AG,1819.317833,5.736827,0.932582,1.834899,64.903112,20.224273,1613.133128,1512.517125,93.762697,...,28.786179,19.899689,8.886490,2.239319,0.446564,1.571663,3.156487,0.413241,0.217797,NaN
4,AG,1932.146133,5.678684,0.863938,1.942463,63.267119,20.577355,2017.073746,1822.171073,90.337355,...,29.254241,20.102809,9.151431,2.196685,0.455231,1.593201,3.177929,0.403733,0.216885,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,RG,1684.869316,6.172513,0.779486,1.762361,66.833116,22.678560,1987.263598,1697.621826,85.425095,...,28.615482,20.899181,7.716302,2.708445,0.369216,1.545084,3.842270,0.255674,0.192777,0.334066
1996,RG,1705.145779,5.849573,0.840795,1.941501,73.520429,19.756656,1789.523101,1615.977103,90.302109,...,23.505766,14.879316,8.626450,1.724848,0.579761,1.024621,2.557620,0.388712,0.321198,0.387368
1997,RG,1533.254111,6.242045,0.788475,1.738466,78.093229,21.369257,1500.003774,1414.638849,94.309019,...,24.546229,16.158504,8.387726,1.926446,0.519090,1.202151,2.932781,0.327634,0.268171,0.386570
1998,RG,1729.859354,5.695733,0.924390,1.944233,71.582641,19.887330,2161.527855,1833.547764,84.826469,...,27.698839,21.277767,6.421071,3.313741,0.301774,1.431245,3.374052,0.295088,0.206808,0.261394


In [ ]:
target=df[['fa_ΣPUFA','group']].copy()

In [ ]:
df_2=pd.read_csv('/content/silurus_glanis_phd_dataset_v2.csv')

In [ ]:
input =df_2[['Group','Water_Temp_C','Water_pH','Water_O2_mgL','Chlorides_mgL','Nitrites_mgL','Nitrates_mgL','Ammonium_mgL','Phosphates_mgL']]

In [ ]:
len(input)

2000

In [ ]:
# Create a copy of target and add a sample_id for merging
df_target_processed = target.copy()
df_target_processed['sample_id'] = df_target_processed.groupby('group').cumcount()

# Create a copy of input, rename 'Group' to 'group', and add a sample_id
df_input_processed = input.copy()
df_input_processed.rename(columns={'Group': 'group'}, inplace=True)
df_input_processed['sample_id'] = df_input_processed.groupby('group').cumcount()

# Merge the two processed dataframes on 'group' and 'sample_id'
merged_data = pd.merge(df_target_processed, df_input_processed, on=['group', 'sample_id'], how='inner')

# Display the head of the merged dataframe
print("Merged DataFrame with fatty acid, environmental, and group information:")
display(merged_data.head())

Merged DataFrame with fatty acid, environmental, and group information:


,fa_ΣPUFA,group,sample_id,Water_Temp_C,Water_pH,Water_O2_mgL,Chlorides_mgL,Nitrites_mgL,Nitrates_mgL,Ammonium_mgL,Phosphates_mgL
0,28.512530,AG,0,23.49,7.710,5.690,79.74,0.1141,1.630,0.1532,0.1714
1,27.820985,AG,1,21.59,7.639,6.327,88.12,0.1391,1.619,0.1821,0.0689
2,28.592778,AG,2,23.94,7.509,5.549,91.69,0.1405,1.082,0.1635,0.1686
3,28.786179,AG,3,26.57,7.403,6.130,110.10,0.1542,1.668,0.1871,0.1307
4,29.254241,AG,4,21.30,7.605,4.228,99.45,0.0990,2.093,0.2072,0.1683


In [ ]:
len(merged_data)

2000

In [ ]:
y=merged_data['fa_ΣPUFA'].copy()

In [ ]:
X=merged_data.drop(columns=['fa_ΣPUFA','group','sample_id'])

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=merged_data['group'])

In [ ]:
X_train_final, X_val, y_train_final, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=merged_data.loc[X_train.index, 'group'])

In [ ]:
print(f"Original training set size: {len(X_train)}")
print(f"Final training set size: {len(X_train_final)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")

Original training set size: 1600
Final training set size: 1200
Validation set size: 400
Test set size: 400


In [ ]:
# """
# ML Training Pipeline — Silurus glanis PhD
# ==========================================
# StandardScaler INSIDE Pipeline → no data leakage in CV or test evaluation.

# Usage:
#     full = prepare_data(static_df, storage_df, phd_df)
#     results, importance = run_all_experiments(full)
# """

# import numpy as np
# import pandas as pd
# from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
# from sklearn.preprocessing import LabelEncoder, StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
# from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,
#                               GradientBoostingRegressor, GradientBoostingClassifier)
# from sklearn.svm import SVR, SVC
# from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
# from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
#                              accuracy_score, f1_score, precision_score, recall_score)
# import xgboost as xgb
# import lightgbm as lgb
# import warnings
# warnings.filterwarnings('ignore')


# def prepare_data(static_df, storage_df, phd_df):
#     """Merge 3 datasets, encode group, drop high-null columns."""
#     full = pd.concat([
#         phd_df,
#         static_df.drop(columns=['group'], errors='ignore'),
#         storage_df.drop(columns=['group'], errors='ignore')
#     ], axis=1)
#     full = full.drop(columns=[c for c in full.columns if full[c].isnull().mean() > 0.3])
#     full = full.fillna(full.median(numeric_only=True))
#     le = LabelEncoder()
#     full['Group_enc'] = le.fit_transform(full['Group'])
#     return full


# def _make_pipelines(task):
#     """
#     All models wrapped in Pipeline([StandardScaler, Model]).
#     Scaler fits ONLY on training data — no leakage.
#     """
#     S = StandardScaler
#     if task == 'reg':
#         return {
#             'Linear Regression': Pipeline([('scaler', S()), ('model', LinearRegression())]),
#             'Ridge':             Pipeline([('scaler', S()), ('model', Ridge(alpha=1.0))]),
#             'Lasso':             Pipeline([('scaler', S()), ('model', Lasso(alpha=0.01))]),
#             'ElasticNet':        Pipeline([('scaler', S()), ('model', ElasticNet(alpha=0.01, l1_ratio=0.5))]),
#             'Random Forest':     Pipeline([('scaler', S()), ('model', RandomForestRegressor(
#                                      n_estimators=200, max_depth=10, min_samples_leaf=5,
#                                      random_state=42, n_jobs=-1))]),
#             'Gradient Boosting': Pipeline([('scaler', S()), ('model', GradientBoostingRegressor(
#                                      n_estimators=200, max_depth=4, learning_rate=0.05,
#                                      subsample=0.8, random_state=42))]),
#             'XGBoost':           Pipeline([('scaler', S()), ('model', xgb.XGBRegressor(
#                                      n_estimators=200, max_depth=4, learning_rate=0.05,
#                                      subsample=0.8, colsample_bytree=0.8,
#                                      reg_alpha=0.1, reg_lambda=1.0,
#                                      random_state=42, verbosity=0))]),
#             'LightGBM':          Pipeline([('scaler', S()), ('model', lgb.LGBMRegressor(
#                                      n_estimators=200, max_depth=4, learning_rate=0.05,
#                                      subsample=0.8, colsample_bytree=0.8,
#                                      reg_alpha=0.1, reg_lambda=1.0,
#                                      random_state=42, verbose=-1))]),
#             'SVR':               Pipeline([('scaler', S()), ('model', SVR(kernel='rbf', C=10.0))]),
#             'KNN':               Pipeline([('scaler', S()), ('model', KNeighborsRegressor(
#                                      n_neighbors=7, weights='distance'))]),
#         }
#     else:
#         return {
#             'Logistic Regression': Pipeline([('scaler', S()), ('model', LogisticRegression(
#                                        max_iter=2000, C=1.0, random_state=42))]),
#             'Random Forest':       Pipeline([('scaler', S()), ('model', RandomForestClassifier(
#                                        n_estimators=200, max_depth=10, random_state=42, n_jobs=-1))]),
#             'Gradient Boosting':   Pipeline([('scaler', S()), ('model', GradientBoostingClassifier(
#                                        n_estimators=200, max_depth=4, learning_rate=0.05,
#                                        random_state=42))]),
#             'XGBoost':             Pipeline([('scaler', S()), ('model', xgb.XGBClassifier(
#                                        n_estimators=200, max_depth=4, learning_rate=0.05,
#                                        random_state=42, verbosity=0, eval_metric='logloss'))]),
#             'LightGBM':            Pipeline([('scaler', S()), ('model', lgb.LGBMClassifier(
#                                        n_estimators=200, max_depth=4, learning_rate=0.05,
#                                        random_state=42, verbose=-1))]),
#             'SVM':                 Pipeline([('scaler', S()), ('model', SVC(
#                                        kernel='rbf', C=10.0, random_state=42))]),
#             'KNN':                 Pipeline([('scaler', S()), ('model', KNeighborsClassifier(
#                                        n_neighbors=7, weights='distance'))]),
#         }


# def run_experiment(full_df, name, features, target, task='reg'):
#     """
#     Train all models for one experiment.
#     StandardScaler inside Pipeline → no leakage in CV folds.
#     """
#     feats = [f for f in features if f in full_df.columns]
#     X = full_df[feats].values
#     y = full_df[target].values

#     # Train/test split (raw data — Pipeline handles scaling)
#     stratify = full_df['Group_enc'] if task == 'cls' else None
#     X_train, X_test, y_train, y_test = train_test_split(
#         X, y, test_size=0.2, random_state=42, stratify=stratify
#     )

#     # CV strategy
#     cv = (KFold(n_splits=5, shuffle=True, random_state=42) if task == 'reg'
#           else StratifiedKFold(n_splits=5, shuffle=True, random_state=42))

#     pipelines = _make_pipelines(task)
#     rows = []

#     for mname, pipe in pipelines.items():
#         # Fit on train only
#         pipe.fit(X_train, y_train)
#         y_pred = pipe.predict(X_test)

#         # CV on full data — Pipeline ensures scaler refits each fold
#         scoring = 'r2' if task == 'reg' else 'accuracy'
#         cv_scores = cross_val_score(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1)

#         if task == 'reg':
#             rows.append({
#                 'experiment': name, 'model': mname,
#                 'R2_test': round(r2_score(y_test, y_pred), 4),
#                 'RMSE': round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
#                 'MAE': round(mean_absolute_error(y_test, y_pred), 4),
#                 'CV_R2_mean': round(cv_scores.mean(), 4),
#                 'CV_R2_std': round(cv_scores.std(), 4),
#             })
#         else:
#             rows.append({
#                 'experiment': name, 'model': mname,
#                 'Accuracy': round(accuracy_score(y_test, y_pred), 4),
#                 'F1': round(f1_score(y_test, y_pred, average='weighted'), 4),
#                 'Precision': round(precision_score(y_test, y_pred, average='weighted'), 4),
#                 'Recall': round(recall_score(y_test, y_pred, average='weighted'), 4),
#                 'CV_Acc_mean': round(cv_scores.mean(), 4),
#                 'CV_Acc_std': round(cv_scores.std(), 4),
#             })

#     # Feature importance (Random Forest)
#     rf_pipe = pipelines['Random Forest']
#     rf_pipe.fit(X, y)
#     imp = pd.DataFrame({
#         'experiment': name,
#         'feature': feats,
#         'importance': rf_pipe.named_steps['model'].feature_importances_
#     }).sort_values('importance', ascending=False).head(10)

#     return pd.DataFrame(rows), imp


# def run_all_experiments(full_df, verbose=True):
#     """Run all 8 experiments, return results + feature importance."""
#     water = ['Water_Temp_C', 'Water_pH', 'Water_O2_mgL', 'Chlorides_mgL',
#              'Nitrites_mgL', 'Nitrates_mgL', 'Ammonium_mgL', 'Phosphates_mgL']
#     fa = [c for c in full_df.columns if c.startswith('fa_')]
#     all_x = (water + ['Group_enc', 'Feed_Type'] +
#              [c for c in full_df.columns
#               if c.startswith(('bio_', 'idx_', 'cut_', 'fa_', 'stor_', 'chem_'))])

#     experiments = [
#         ('R1_Water→Protein',  water + ['Group_enc', 'Feed_Type'], 'Protein_perc',  'reg'),
#         ('R2_Water→Lipids',   water + ['Group_enc', 'Feed_Type'], 'Lipids_perc',   'reg'),
#         ('R3_Water→Bodymass', water + ['Group_enc', 'Feed_Type'], 'Bodymass_g',    'reg'),
#         ('R4_All→Protein',    all_x,                              'Protein_perc',  'reg'),
#         ('R5_All→Lipids',     all_x,                              'Lipids_perc',   'reg'),
#         ('C1_Water→Group',    water + ['Feed_Type'],              'Group_enc',     'cls'),
#         ('C2_FA→Group',       fa,                                 'Group_enc',     'cls'),
#         ('C3_All→Group',      all_x,                              'Group_enc',     'cls'),
#     ]

#     all_results, all_imps = [], []
#     for name, feats, target, task in experiments:
#         if verbose:
#             print(f"\n{'='*55}\n{name}")
#         res, imp = run_experiment(full_df, name, feats, target, task)
#         all_results.append(res)
#         all_imps.append(imp)
#         if verbose:
#             for _, r in res.iterrows():
#                 if task == 'reg':
#                     print(f"  {r['model']:25s} R²={r['R2_test']:7.4f}  "
#                           f"RMSE={r['RMSE']:8.4f}  "
#                           f"CV_R²={r['CV_R2_mean']:.4f}±{r['CV_R2_std']:.4f}")
#                 else:
#                     print(f"  {r['model']:25s} Acc={r['Accuracy']:.4f}  "
#                           f"F1={r['F1']:.4f}  "
#                           f"CV={r['CV_Acc_mean']:.4f}±{r['CV_Acc_std']:.4f}")

#     return pd.concat(all_results, ignore_index=True), pd.concat(all_imps, ignore_index=True)

### Linear Regression Model Training and Evaluation

In [ ]:
# from sklearn.linear_model import LinearRegression
# from sklearn.metrics import mean_squared_error, r2_score
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline

# # Initialize and train the Linear Regression model with a pipeline for scaling
# pipeline = Pipeline([
#     ('scaler', StandardScaler()),
#     ('regressor', LinearRegression(n_))
# ])

# pipeline.fit(X_train_final, y_train_final)

# print("Linear Regression model with StandardScaler trained successfully.")

NameError: name 'n_' is not defined

In [ ]:
# # Make predictions on the validation set using the pipeline
# y_pred_val = pipeline.predict(X_val)

# # Evaluate the model on the validation set
# mse_val = mean_squared_error(y_val, y_pred_val)
# r2_val = r2_score(y_val, y_pred_val)

# print(f"Validation Mean Squared Error (MSE): {mse_val:.4f}")
# print(f"Validation R-squared (R2): {r2_val:.4f}")

Validation Mean Squared Error (MSE): 13.7478
Validation R-squared (R2): -0.3563


### Feature Importances (Coefficients)

In [ ]:
# import pandas as pd

# # Get the Linear Regression regressor from the pipeline
# linear_regressor = pipeline.named_steps['regressor']

# # Get the coefficients (weights) of the features from the regressor
# feature_importances = pd.DataFrame({
#     'Feature': X_train_final.columns,
#     'Coefficient': linear_regressor.coef_
# })

# # Sort by absolute coefficient value to see most impactful features
# feature_importances['Abs_Coefficient'] = abs(feature_importances['Coefficient'])
# feature_importances = feature_importances.sort_values(by='Abs_Coefficient', ascending=False)

# display(feature_importances)

### RandomForestRegressor Model Training and Evaluation

In [ ]:
# from sklearn.ensemble import RandomForestRegressor

# # Initialize and train the RandomForestRegressor model with a pipeline for scaling
# # Using a small number of estimators for quicker demonstration, consider increasing for production
# rfc_pipeline = Pipeline([

#     ('regressor', RandomForestRegressor(random_state=42))
# ])

# rfc_pipeline.fit(X_train_final, y_train_final)

# print("RandomForestRegressor model with StandardScaler trained successfully.")

In [ ]:
# # Make predictions on the validation set using the pipeline
# y_pred_val_rfc = rfc_pipeline.predict(X_val)

# # Evaluate the model on the validation set
# mse_val_rfc = mean_squared_error(y_val, y_pred_val_rfc)
# r2_val_rfc = r2_score(y_val, y_pred_val_rfc)

# print(f"RandomForest Validation Mean Squared Error (MSE): {mse_val_rfc:.4f}")
# print(f"RandomForest Validation R-squared (R2): {r2_val_rfc:.4f}")

### RandomForest Feature Importances

In [ ]:
# # Get the RandomForest regressor from the pipeline
# rfc_regressor = rfc_pipeline.named_steps['regressor']

# # Get the feature importances from the regressor
# rfc_feature_importances = pd.DataFrame({
#     'Feature': X_train_final.columns,
#     'Importance': rfc_regressor.feature_importances_
# })

# # Sort by importance value
# rfc_feature_importances = rfc_feature_importances.sort_values(by='Importance', ascending=False)

# display(rfc_feature_importances)

### Hyperparameter Tuning for RandomForestRegressor using GridSearchCV

In [ ]:
# from sklearn.model_selection import GridSearchCV

# # Define the parameter grid to search
# param_grid = {
#     'regressor__n_estimators': [50, 100, 200],  # Number of trees in the forest
#     'regressor__max_depth': [None, 10, 20],   # Maximum depth of the tree
#     'regressor__min_samples_split': [2, 5],  # Minimum number of samples required to split an internal node
#     'regressor__min_samples_leaf': [1, 2]     # Minimum number of samples required to be at a leaf node
# }

# # Create a GridSearchCV object
# grid_search = GridSearchCV(
#     estimator=rfc_pipeline, # Use the existing pipeline with the StandardScaler
#     param_grid=param_grid,
#     cv=3,  # Number of cross-validation folds
#     scoring='r2', # Metric to optimize for (R-squared)
#     n_jobs=-1, # Use all available cores
#     verbose=2
# )

# # Fit GridSearchCV to the training data
# grid_search.fit(X_train_final, y_train_final)

# print("GridSearchCV completed.")

In [ ]:
# # Print the best parameters and best score
# print(f"Best parameters found: {grid_search.best_params_}")
# print(f"Best R-squared score found: {grid_search.best_score_:.4f}")

# # Get the best estimator (the pipeline with optimal parameters)
# best_rfc_pipeline = grid_search.best_estimator_

# # Make predictions on the validation set using the best pipeline
# y_pred_val_best_rfc = best_rfc_pipeline.predict(X_val)

# # Evaluate the best model on the validation set
# mse_val_best_rfc = mean_squared_error(y_val, y_pred_val_best_rfc)
# r2_val_best_rfc = r2_score(y_val, y_pred_val_best_rfc)

# print(f"\nBest RandomForest Validation Mean Squared Error (MSE): {mse_val_best_rfc:.4f}")
# print(f"Best RandomForest Validation R-squared (R2): {r2_val_best_rfc:.4f}")

### Feature Importances from Best RandomForest Model

In [ ]:
# # Get the best RandomForest regressor from the pipeline
# best_rfc_regressor = best_rfc_pipeline.named_steps['regressor']

# # Get the feature importances from the best regressor
# best_rfc_feature_importances = pd.DataFrame({
#     'Feature': X_train_final.columns,
#     'Importance': best_rfc_regressor.feature_importances_
# })

# # Sort by importance value
# best_rfc_feature_importances = best_rfc_feature_importances.sort_values(by='Importance', ascending=False)

# display(best_rfc_feature_importances)

### PyTorch MLP Model Training and Evaluation

In [ ]:
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import TensorDataset, DataLoader
# from sklearn.preprocessing import StandardScaler

# # 1. Prepare data for PyTorch

# # Scale the features
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train_final)
# X_val_scaled = scaler.transform(X_val)

# # Convert numpy arrays to PyTorch tensors
# X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
# y_train_tensor = torch.tensor(y_train_final.values, dtype=torch.float32).unsqueeze(1)
# X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
# y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)

# # Create TensorDatasets
# train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
# val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

# # Create DataLoaders for batching
# batch_size = 64
# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# print("Data prepared and loaded into PyTorch DataLoaders.")

Data prepared and loaded into PyTorch DataLoaders.


In [ ]:
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import TensorDataset, DataLoader
# from sklearn.preprocessing import StandardScaler

# # 2. Define the MLP Model
# class MLP(nn.Module):
#     def __init__(self, input_size):
#         super(MLP, self).__init__()
#         self.layer1 = nn.Linear(input_size, 128)
#         self.relu1 = nn.ReLU()
#         self.layer2 = nn.Linear(128, 64)
#         self.relu2 = nn.ReLU()
#         self.layer3 = nn.Linear(64, 32)
#         self.relu3 = nn.ReLU()
#         self.layer4 = nn.Linear(32, 16) # Added new layer
#         self.relu4 = nn.ReLU()
#         self.output_layer = nn.Linear(16, 1) # Single output for regression

#     def forward(self, x):
#         x = self.relu1(self.layer1(x))
#         x = self.relu2(self.layer2(x))
#         x = self.relu3(self.layer3(x))
#         x = self.relu4(self.layer4(x)) # Forward pass for new layer
#         x = self.output_layer(x)
#         return x

# # Initialize the model
# input_size = X_train_final.shape[1] # Number of features
# mlp_model = MLP(input_size)

# # Define Loss Function and Optimizer
# criterion = nn.MSELoss() # Mean Squared Error for regression
# optimizer = optim.Adam(mlp_model.parameters(), lr=0.001)

# print(f"MLP model initialized with {input_size} input features.")
# print(mlp_model)

MLP model initialized with 8 input features.
MLP(
  (layer1): Linear(in_features=8, out_features=128, bias=True)
  (relu1): ReLU()
  (layer2): Linear(in_features=128, out_features=64, bias=True)
  (relu2): ReLU()
  (layer3): Linear(in_features=64, out_features=32, bias=True)
  (relu3): ReLU()
  (layer4): Linear(in_features=32, out_features=16, bias=True)
  (relu4): ReLU()
  (output_layer): Linear(in_features=16, out_features=1, bias=True)
)


In [ ]:
# # 3. Training Loop
# num_epochs = 1000  # Increased epochs to allow for early stopping
# patience = 50      # Number of epochs to wait after last improvement
# min_val_loss = float('inf')
# epochs_no_improve = 0
# best_model_state = None

# for epoch in range(num_epochs):
#     mlp_model.train() # Set model to training mode
#     for inputs, labels in train_loader:
#         optimizer.zero_grad() # Zero the gradients
#         outputs = mlp_model(inputs)
#         loss = criterion(outputs, labels)
#         loss.backward() # Perform backpropagation
#         optimizer.step() # Update weights

#     # Validation phase
#     mlp_model.eval() # Set model to evaluation mode
#     val_losses = []
#     with torch.no_grad(): # Disable gradient calculation for evaluation
#         for inputs, labels in val_loader:
#             outputs = mlp_model(inputs)
#             val_loss = criterion(outputs, labels)
#             val_losses.append(val_loss.item())

#     avg_val_loss = sum(val_losses) / len(val_losses)
#     print(f'Epoch [{epoch+1}/{num_epochs}], Training Loss: {loss.item():.4f}, Validation Loss: {avg_val_loss:.4f}')

#     # Early stopping logic
#     if avg_val_loss < min_val_loss:
#         min_val_loss = avg_val_loss
#         epochs_no_improve = 0
#         best_model_state = mlp_model.state_dict() # Save best model state
#     else:
#         epochs_no_improve += 1
#         if epochs_no_improve >= patience:
#             print(f'Early stopping triggered after {patience} epochs without improvement.')
#             break

# # Load the best model state after training (or early stopping)
# if best_model_state:
#     mlp_model.load_state_dict(best_model_state)
#     print("Loaded best model state.")

# print("Training complete.")

Epoch [1/1000], Training Loss: 749.2792, Validation Loss: 707.3747
Epoch [2/1000], Training Loss: 633.0488, Validation Loss: 606.3283
Epoch [3/1000], Training Loss: 229.5766, Validation Loss: 209.6834
Epoch [4/1000], Training Loss: 62.4327, Validation Loss: 60.7377
Epoch [5/1000], Training Loss: 60.9222, Validation Loss: 38.1225
Epoch [6/1000], Training Loss: 23.2798, Validation Loss: 32.0765
Epoch [7/1000], Training Loss: 26.9258, Validation Loss: 28.3240
Epoch [8/1000], Training Loss: 33.7424, Validation Loss: 25.6109
Epoch [9/1000], Training Loss: 21.7107, Validation Loss: 22.7732
Epoch [10/1000], Training Loss: 32.6892, Validation Loss: 20.3442
Epoch [11/1000], Training Loss: 19.4426, Validation Loss: 18.2715
Epoch [12/1000], Training Loss: 28.2387, Validation Loss: 16.2913
Epoch [13/1000], Training Loss: 16.3972, Validation Loss: 14.7861
Epoch [14/1000], Training Loss: 16.3317, Validation Loss: 13.7478
Epoch [15/1000], Training Loss: 10.3298, Validation Loss: 12.4035
Epoch [16/100

In [ ]:
# from sklearn.metrics import r2_score
# import numpy as np

# # 4. Evaluate the trained MLP model on the validation set
# mlp_model.eval() # Set model to evaluation mode
# all_predictions = []
# all_true_labels = []

# with torch.no_grad():
#     for inputs, labels in val_loader:
#         outputs = mlp_model(inputs)
#         all_predictions.extend(outputs.numpy().flatten())
#         all_true_labels.extend(labels.numpy().flatten())

# # Calculate MSE and R2 on the validation set
# mse_val_mlp = mean_squared_error(all_true_labels, all_predictions)
# r2_val_mlp = r2_score(all_true_labels, all_predictions)

# print(f"\nMLP Validation Mean Squared Error (MSE): {mse_val_mlp:.4f}")
# print(f"MLP Validation R-squared (R2): {r2_val_mlp:.4f}")


MLP Validation Mean Squared Error (MSE): 6.6074
MLP Validation R-squared (R2): 0.3481


In [ ]:
"""
Full ML Pipeline — Silurus glanis PhD
======================================
1. Generate v2 dataset (water↔fish correlations)
2. Merge with synthetic static + storage
3. Train models with Pipeline(StandardScaler → Model) — NO data leakage
4. Save results

Usage:
    python ml_full_pipeline.py
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                             accuracy_score, f1_score, precision_score, recall_score)
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')


# ══════════════════════════════════════════════════════════════════
# STEP 1: Generate v2 PHD dataset with biological correlations
# ══════════════════════════════════════════════════════════════════

def generate_phd_dataset(n_per_group=1000, seed=42):
    """
    Generate water quality → fish quality dataset with real biological
    correlations based on S. glanis literature.

    Relationships encoded:
        Temp  → Lipids (+)  : Hallier et al. (2007) Food Chem. 103:808
        O2    → Protein (+) : Huss (1995) FAO Fish. Tech. Paper 348
        Temp  → Bodymass (+): Simeanu et al. (2022) Agriculture 12:2144
        NH4+  → Protein (-) : ammonia toxicity, protein catabolism
        Feed  → Lipids (+)  : Jankowska et al. (2007) Eur. Food Res. Technol.
    """
    rng = np.random.default_rng(seed)

    def _gen(group, n):
        ag = group == 'AG'
        temp = np.clip(rng.normal(22 if ag else 12, 3, n), 8, 30)
        pH = np.clip(rng.normal(7.5 if ag else 7.3, 0.15, n), 6.8, 8.2)
        O2 = np.clip(rng.normal(6.5 if ag else 9.0, 1.2, n), 3.5, 12)
        cl = np.clip(rng.normal(95 if ag else 75, 8, n), 50, 120)
        no2 = np.clip(rng.normal(0.14 if ag else 0.08, 0.03, n), 0.01, 0.25)
        no3 = np.clip(rng.normal(1.8 if ag else 1.0, 0.4, n), 0.1, 3.0)
        nh4 = np.clip(rng.normal(0.22 if ag else 0.10, 0.06, n), 0.02, 0.40)
        po4 = np.clip(rng.normal(0.14 if ag else 0.06, 0.04, n), 0.001, 0.25)
        feed = np.ones(n) if ag else np.zeros(n)

        mass = np.clip(1500 + 12*temp + 15*O2 - 2.5*cl + 300*feed
                       + rng.normal(0, 120, n), 1200, 2400)
        prot = np.clip(15.0 + 0.35*O2 - 0.12*temp - 8.0*nh4 + 5.0*no2
                       + 0.5*feed + rng.normal(0, 1.5, n), 10, 25)
        lip = np.clip(1.5 + 0.06*temp + 0.8*feed - 0.08*O2 + 3.0*po4
                      - 1.5*no2 + rng.normal(0, 0.3, n), 1.0, 5.5)

        return pd.DataFrame({
            'Group': group, 'Water_Temp_C': temp.round(2),
            'Water_pH': pH.round(3), 'Water_O2_mgL': O2.round(3),
            'Chlorides_mgL': cl.round(2), 'Nitrites_mgL': no2.round(4),
            'Nitrates_mgL': no3.round(3), 'Ammonium_mgL': nh4.round(4),
            'Phosphates_mgL': po4.round(4), 'Feed_Type': feed.astype(int),
            'Bodymass_g': mass.round(2),
            'Protein_perc': prot.round(3), 'Lipids_perc': lip.round(3),
        })

    df = pd.concat([_gen('AG', n_per_group), _gen('RG', n_per_group)],
                   ignore_index=True)
    df.insert(0, 'Fish_ID', range(1, len(df) + 1))
    return df


# ══════════════════════════════════════════════════════════════════
# STEP 2: Merge datasets
# ══════════════════════════════════════════════════════════════════

def prepare_full_dataset(phd_df, static_df, storage_df):
    """Merge PHD + static + storage, encode group, handle nulls."""
    full = pd.concat([
        phd_df,
        static_df.drop(columns=['group'], errors='ignore'),
        storage_df.drop(columns=['group'], errors='ignore')
    ], axis=1)
    full = full.drop(columns=[c for c in full.columns
                               if full[c].isnull().mean() > 0.3])
    full = full.fillna(full.median(numeric_only=True))
    le = LabelEncoder()
    full['Group_enc'] = le.fit_transform(full['Group'])
    return full


# ══════════════════════════════════════════════════════════════════
# STEP 3: Model training with Pipeline (no leakage)
# ══════════════════════════════════════════════════════════════════

def _pipelines(task):
    """StandardScaler INSIDE Pipeline → fits only on train fold."""
    S = StandardScaler
    if task == 'reg':
        return {
            'Linear Regression': Pipeline([('s', S()), ('m', LinearRegression())]),
            'Ridge':             Pipeline([('s', S()), ('m', Ridge(alpha=1.0))]),
            'Lasso':             Pipeline([('s', S()), ('m', Lasso(alpha=0.01))]),
            'Random Forest':     Pipeline([('s', S()), ('m', RandomForestRegressor(
                n_estimators=150, max_depth=8, min_samples_leaf=5,
                random_state=42, n_jobs=-1))]),
            'XGBoost':           Pipeline([('s', S()), ('m', xgb.XGBRegressor(
                n_estimators=150, max_depth=4, learning_rate=0.05,
                subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                random_state=42, verbosity=0))]),
            'LightGBM':          Pipeline([('s', S()), ('m', lgb.LGBMRegressor(
                n_estimators=150, max_depth=4, learning_rate=0.05,
                subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                random_state=42, verbose=-1))]),
            'SVR':               Pipeline([('s', S()), ('m', SVR(kernel='rbf', C=10.0))]),
            'KNN':               Pipeline([('s', S()), ('m', KNeighborsRegressor(
                n_neighbors=7, weights='distance'))]),
        }
    else:
        return {
            'Logistic Regression': Pipeline([('s', S()), ('m', LogisticRegression(
                max_iter=2000, random_state=42))]),
            'Random Forest':       Pipeline([('s', S()), ('m', RandomForestClassifier(
                n_estimators=150, max_depth=8, random_state=42, n_jobs=-1))]),
            'XGBoost':             Pipeline([('s', S()), ('m', xgb.XGBClassifier(
                n_estimators=150, max_depth=4, learning_rate=0.05,
                random_state=42, verbosity=0, eval_metric='logloss'))]),
            'LightGBM':            Pipeline([('s', S()), ('m', lgb.LGBMClassifier(
                n_estimators=150, max_depth=4, learning_rate=0.05,
                random_state=42, verbose=-1))]),
            'SVM':                 Pipeline([('s', S()), ('m', SVC(
                kernel='rbf', C=10.0, random_state=42))]),
            'KNN':                 Pipeline([('s', S()), ('m', KNeighborsClassifier(
                n_neighbors=7, weights='distance'))]),
        }


def run_experiment(full_df, name, features, target, task='reg'):
    """Train all models, evaluate on holdout + 5-fold CV."""
    feats = [f for f in features if f in full_df.columns]
    X, y = full_df[feats].values, full_df[target].values

    stratify = full_df['Group_enc'] if task == 'cls' else None
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=stratify)

    cv = (KFold(5, shuffle=True, random_state=42) if task == 'reg'
          else StratifiedKFold(5, shuffle=True, random_state=42))

    pipes = _pipelines(task)
    rows = []

    for mname, pipe in pipes.items():
        pipe.fit(X_tr, y_tr)
        yp = pipe.predict(X_te)
        scoring = 'r2' if task == 'reg' else 'accuracy'
        cvs = cross_val_score(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1)

        if task == 'reg':
            rows.append({
                'experiment': name, 'model': mname,
                'R2_test': round(r2_score(y_te, yp), 4),
                'RMSE': round(np.sqrt(mean_squared_error(y_te, yp)), 4),
                'MAE': round(mean_absolute_error(y_te, yp), 4),
                'CV_R2_mean': round(cvs.mean(), 4),
                'CV_R2_std': round(cvs.std(), 4),
            })
        else:
            rows.append({
                'experiment': name, 'model': mname,
                'Accuracy': round(accuracy_score(y_te, yp), 4),
                'F1': round(f1_score(y_te, yp, average='weighted'), 4),
                'Precision': round(precision_score(y_te, yp, average='weighted'), 4),
                'Recall': round(recall_score(y_te, yp, average='weighted'), 4),
                'CV_Acc_mean': round(cvs.mean(), 4),
                'CV_Acc_std': round(cvs.std(), 4),
            })

    # Feature importance (RF)
    rf = pipes['Random Forest']
    rf.fit(X, y)
    imp = pd.DataFrame({
        'experiment': name, 'feature': feats,
        'importance': rf.named_steps['m'].feature_importances_
    }).sort_values('importance', ascending=False).head(10)

    return pd.DataFrame(rows), imp


def run_all(full_df, verbose=True):
    """Run all 8 experiments."""
    w = ['Water_Temp_C', 'Water_pH', 'Water_O2_mgL', 'Chlorides_mgL',
         'Nitrites_mgL', 'Nitrates_mgL', 'Ammonium_mgL', 'Phosphates_mgL']
    fa = [c for c in full_df.columns if c.startswith('fa_')]
    allx = (w + ['Group_enc', 'Feed_Type'] +
            [c for c in full_df.columns
             if c.startswith(('bio_', 'idx_', 'cut_', 'fa_', 'stor_', 'chem_'))])

    exps = [
        ('R1_Water→Protein',  w + ['Group_enc', 'Feed_Type'], 'Protein_perc',  'reg'),
        ('R2_Water→Lipids',   w + ['Group_enc', 'Feed_Type'], 'Lipids_perc',   'reg'),
        ('R3_Water→Bodymass', w + ['Group_enc', 'Feed_Type'], 'Bodymass_g',    'reg'),
        ('R4_All→Protein',    allx,                           'Protein_perc',  'reg'),
        ('R5_All→Lipids',     allx,                           'Lipids_perc',   'reg'),
        ('C1_Water→Group',    w + ['Feed_Type'],              'Group_enc',     'cls'),
        ('C2_FA→Group',       fa,                             'Group_enc',     'cls'),
        ('C3_All→Group',      allx,                           'Group_enc',     'cls'),
    ]

    all_res, all_imp = [], []
    for name, feats, target, task in exps:
        if verbose:
            print(f"\n{'='*55}\n{name}")
        r, i = run_experiment(full_df, name, feats, target, task)
        all_res.append(r)
        all_imp.append(i)
        if verbose:
            for _, row in r.iterrows():
                if task == 'reg':
                    print(f"  {row['model']:25s} R²={row['R2_test']:7.4f}  "
                          f"RMSE={row['RMSE']:8.4f}  "
                          f"CV_R²={row['CV_R2_mean']:.4f}±{row['CV_R2_std']:.4f}")
                else:
                    print(f"  {row['model']:25s} Acc={row['Accuracy']:.4f}  "
                          f"F1={row['F1']:.4f}  "
                          f"CV={row['CV_Acc_mean']:.4f}±{row['CV_Acc_std']:.4f}")

    return (pd.concat(all_res, ignore_index=True),
            pd.concat(all_imp, ignore_index=True))


# ══════════════════════════════════════════════════════════════════
# STEP 4: Main
# ══════════════════════════════════════════════════════════════════


    # Generate / load
phd = generate_phd_dataset(n_per_group=1000, seed=42)
static = pd.read_csv('synthetic_static.csv')
storage = pd.read_csv('synthetic_storage.csv')

    # Merge
full = prepare_full_dataset(phd, static, storage)
print(f"Dataset: {full.shape}")

    # Train
results, importance = run_all(full, verbose=True)

    # Save
phd.to_csv('silurus_glanis_phd_dataset_v2.csv', index=False)
results.to_csv('ml_results_v2.csv', index=False)
importance.to_csv('feature_importance_v2.csv', index=False)

print(f"\n✓ results: {results.shape}")
print(f"✓ importance: {importance.shape}")

Dataset: (2000, 85)

R1_Water→Protein


KeyboardInterrupt: 

In [ ]:
"""
Transfer Learning Validation: Silurus glanis → Acipenser spp.
==============================================================

Models trained on S. glanis synthetic water quality → flesh composition data
are applied to predict flesh lipid/protein content of three sturgeon species
using published water quality and proximate composition data.

Sturgeon test cases (peer-reviewed sources):
    1. Acipenser stellatus — Florescu Gune et al. (2021); Dorojan et al. (2020)
    2. Acipenser baerii — Lopez et al. (2020)
    3. Huso huso — Ghomi et al. (2013)
"""

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')


def run_transfer_test(phd_df, static_df, storage_df):
    """
    Train on Silurus glanis, predict sturgeon, return results DataFrame.

    StandardScaler inside Pipeline — no data leakage.
    """

    # ── Prepare training data ──
    full = pd.concat([
        phd_df,
        static_df.drop(columns=['group'], errors='ignore'),
        storage_df.drop(columns=['group'], errors='ignore')
    ], axis=1)
    full = full.drop(columns=[c for c in full.columns if full[c].isnull().mean() > 0.3])
    full = full.fillna(full.median(numeric_only=True))
    le = LabelEncoder()
    full['Group_enc'] = le.fit_transform(full['Group'])

    features = [
        'Water_Temp_C', 'Water_pH', 'Water_O2_mgL', 'Chlorides_mgL',
        'Nitrites_mgL', 'Nitrates_mgL', 'Ammonium_mgL', 'Phosphates_mgL',
        'Group_enc', 'Feed_Type'
    ]

    X_train = full[features].values
    targets = {
        'Lipids (%)': full['Lipids_perc'].values,
        'Protein (%)': full['Protein_perc'].values,
        'Body mass (g)': full['Bodymass_g'].values,
    }

    S = StandardScaler
    model_defs = [
        ('Ridge', lambda: Pipeline([('s', S()), ('m', Ridge(alpha=1.0))])),
        ('Random Forest', lambda: Pipeline([('s', S()), ('m', RandomForestRegressor(
            n_estimators=150, max_depth=8, random_state=42, n_jobs=-1))])),
        ('LightGBM', lambda: Pipeline([('s', S()), ('m', lgb.LGBMRegressor(
            n_estimators=150, max_depth=4, learning_rate=0.05,
            random_state=42, verbose=-1))])),
    ]

    # Train
    models = {}
    for tname, y in targets.items():
        for mname, mfn in model_defs:
            m = mfn()
            m.fit(X_train, y)
            models[(tname, mname)] = m

    # ── Sturgeon test cases ──
    sturgeons = [
        {
            'name': 'Acipenser stellatus',
            'common': 'Sevruga sturgeon',
            'source': 'RAS, Galați, Romania',
            'water_ref': 'Florescu Gune et al. (2021)',
            'flesh_ref': 'Dorojan et al. (2020)',
            'Water_Temp_C': 25.0, 'Water_pH': 7.70, 'Water_O2_mgL': 6.60,
            'Chlorides_mgL': 90.0, 'Nitrites_mgL': 0.32,
            'Nitrates_mgL': 32.20, 'Ammonium_mgL': 0.43,
            'Phosphates_mgL': 0.10, 'Group_enc': 0, 'Feed_Type': 1,
            'actual': {
                'Lipids (%)': 1.32,
                'Protein (%)': 17.86,
                'Body mass (g)': 331.3,
            },
        },
        {
            'name': 'Acipenser baerii',
            'common': 'Siberian sturgeon',
            'source': 'Farm, Agroittica Lombarda, Italy',
            'water_ref': 'Estimated from facility reports',
            'flesh_ref': 'Lopez et al. (2020)',
            'Water_Temp_C': 18.0, 'Water_pH': 7.40, 'Water_O2_mgL': 8.50,
            'Chlorides_mgL': 80.0, 'Nitrites_mgL': 0.10,
            'Nitrates_mgL': 5.0, 'Ammonium_mgL': 0.15,
            'Phosphates_mgL': 0.08, 'Group_enc': 0, 'Feed_Type': 1,
            'actual': {
                'Lipids (%)': 5.60,
                'Protein (%)': 17.60,
                'Body mass (g)': 6500.0,
            },
        },
        {
            'name': 'Huso huso',
            'common': 'Beluga sturgeon',
            'source': 'Concrete tanks, Sari, Iran',
            'water_ref': 'Estimated from RAS standards',
            'flesh_ref': 'Ghomi et al. (2013)',
            'Water_Temp_C': 22.0, 'Water_pH': 7.50, 'Water_O2_mgL': 7.00,
            'Chlorides_mgL': 95.0, 'Nitrites_mgL': 0.15,
            'Nitrates_mgL': 2.0, 'Ammonium_mgL': 0.20,
            'Phosphates_mgL': 0.12, 'Group_enc': 0, 'Feed_Type': 1,
            'actual': {
                'Lipids (%)': 3.92,
                'Protein (%)': 14.73,
                'Body mass (g)': 15000.0,
            },
        },
    ]

    # ── Predict & compare ──
    rows = []
    for ex in sturgeons:
        X_test = np.array([[ex[f] for f in features]])
        for tname in targets:
            actual = ex['actual'][tname]
            for mname, _ in model_defs:
                pred = models[(tname, mname)].predict(X_test)[0]
                err = pred - actual
                pct = (err / actual) * 100 if actual != 0 else 0
                rows.append({
                    'Species': ex['name'],
                    'Common Name': ex['common'],
                    'Rearing System': ex['source'],
                    'Water Quality Ref': ex['water_ref'],
                    'Flesh Composition Ref': ex['flesh_ref'],
                    'Target': tname,
                    'Model': mname,
                    'Actual': round(actual, 2),
                    'Predicted': round(pred, 2),
                    'Absolute Error': round(abs(err), 2),
                    'Error (%)': round(pct, 1),
                })

    return pd.DataFrame(rows)

In [ ]:
phd_df=phd

In [ ]:
results = run_transfer_test(phd_df, static_df, storage_df)
results.to_csv('transfer_learning_results.csv', index=False)